# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Haryomhidhe/Machine-learning-flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule: A page is scored as an opportunity only if it has real search demand (impressions > 100 for the month). Among those pages, if CTR is below 2%, that means people see it but don't click, so it gets flagged as low_ctr_visible_page. If avg_position is worse than 10, the page isn't ranking well despite demand, so it gets flagged as poor_position. If both apply, it's the highest priority opportunity, flagged as low_ctr_and_poor_position. Pages with real demand that don't hit either problem are performing_well and left alone. Pages with low demand (under 100 impressions) are low_demand, meaning there isn't enough traffic yet to justify reviewing them

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df_score = con.sql(f"""
SELECT
  content_hash_id,
  SUM(gsc_impressions) as total_impressions,
  SUM(gsc_clicks) as total_clicks,
  AVG(gsc_avg_position) as avg_position,
  CASE WHEN SUM(gsc_impressions) > 0
    THEN (SUM(gsc_clicks)::FLOAT / SUM(gsc_impressions)) * 100
    ELSE 0 END as ctr
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
GROUP BY content_hash_id
""").df()

def score_page(row):
    if row['total_impressions'] <= 100:
        return 0, 'low_demand', 'leave_as_is'

    low_ctr = row['ctr'] < 2.0
    poor_position = row['avg_position'] > 10

    if not low_ctr and not poor_position:
        return 0, 'performing_well', 'leave_as_is'

    severity = 0
    if low_ctr:
        severity += 2
    if poor_position:
        severity += 1

    score = row['total_impressions'] * severity

    if low_ctr and poor_position:
        reason = 'low_ctr_and_poor_position'
        action = 'review_and_refresh'
    elif low_ctr:
        reason = 'low_ctr_visible_page'
        action = 'review_ctr'
    else:
        reason = 'poor_position'
        action = 'review_position'

    return score, reason, action

df_score[['action_score', 'reason_code', 'suggested_action']] = df_score.apply(
    lambda row: pd.Series(score_page(row)), axis=1
)

df_score = df_score.sort_values('action_score', ascending=False)

import os
os.makedirs('work/outputs', exist_ok=True)
df_score.to_csv('work/outputs/baseline_action_score.csv', index=False)

df_score.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,total_impressions,total_clicks,avg_position,ctr,action_score,reason_code,suggested_action
22248,content_e8a52cf3d5988c07,162129.0,627.0,13.725133,0.386729,486387.0,low_ctr_and_poor_position,review_and_refresh
210568,content_8e1334d6356668e3,203401.0,2.0,4.967059,0.000983,406802.0,low_ctr_visible_page,review_ctr
221928,content_9c057b66c30a3abb,195648.0,1.0,2.488437,0.000511,391296.0,low_ctr_visible_page,review_ctr
50000,content_fec55986a1868d62,193954.0,0.0,3.844678,0.000000,387908.0,low_ctr_visible_page,review_ctr
222919,content_512dbad65bd5ade9,167303.0,3310.0,2.923168,1.978446,334606.0,low_ctr_visible_page,review_ctr
271343,content_e241d6415ac9e534,164152.0,401.0,2.925926,0.244286,328304.0,low_ctr_visible_page,review_ctr
92324,content_b99ea6861864dea5,160699.0,273.0,3.715542,0.169883,321398.0,low_ctr_visible_page,review_ctr
122443,content_f107e54b10b43725,156163.0,883.0,3.095657,0.565435,312326.0,low_ctr_visible_page,review_ctr
36790,content_e7b5dd4dff461ad2,154502.0,2508.0,4.240872,1.623280,309004.0,low_ctr_visible_page,review_ctr
183546,content_36e53e9c707674fc,100736.0,237.0,34.040988,0.235268,302208.0,low_ctr_and_poor_position,review_and_refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.